In [1]:
import torch
import torch.nn as nn

In [2]:
layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [4]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(50, 40), nn.ReLU(), nn.Linear(40, 1), nn.ReLU())
model.apply(use_he_init)

Sequential(
  (0): Linear(in_features=50, out_features=40, bias=True)
  (1): ReLU()
  (2): Linear(in_features=40, out_features=1, bias=True)
  (3): ReLU()
)

In [5]:
alpha = 0.2
model = nn.Sequential(nn.Linear(50, 40), nn.LeakyReLU(negative_slope=alpha))
nn.init.kaiming_uniform_(model[0].weight, alpha, nonlinearity="leaky_relu")

Parameter containing:
tensor([[ 0.0026,  0.2796,  0.2112,  ...,  0.1501,  0.1676,  0.2458],
        [ 0.0998,  0.0761,  0.0969,  ...,  0.1303,  0.0420,  0.0835],
        [ 0.2661, -0.2731,  0.3154,  ...,  0.0403, -0.0292,  0.0269],
        ...,
        [-0.2376, -0.1699,  0.1564,  ...,  0.0817,  0.0828, -0.3363],
        [ 0.0326, -0.1592,  0.0504,  ...,  0.0729,  0.0475, -0.1847],
        [ 0.1508,  0.0262, -0.0435,  ...,  0.0004,  0.0525,  0.0085]],
       requires_grad=True)

In [6]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28, 300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10),
)

In [7]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [8]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [9]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

In [10]:
inputs = torch.randn(32, 3, 100, 200)
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)

In [11]:
means = inputs.mean(dim=[2, 3], keepdim=True)
vars_ = inputs.var(dim=[2, 3], keepdim=True, unbiased=False)
stds = torch.sqrt(vars_ + layer_norm.eps)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [12]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [13]:
#Gradiemt clipping

# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         X_batch, y_batch = X_batch.to(device), y_batch.to(device)
#         y_pred = model(X_batch)
#         loss = loss_fn(y_pred, y_batch)
#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         optimizer.step()
#         optimizer.zero_grad()

In [14]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8),
)

# train this model or load pretrained weights

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
import copy

torch.manual_seed(42)
reused_layers = copy.deepcopy(model_A[:-1])
model_B_on_A = nn.Sequential(
    *reused_layers,
    nn.Linear(100, 1)
).to(device)

In [17]:
for layer in model_B_on_A[:-1]:
    for param in layer.parameters():
        param.requires_grad = False

In [18]:
import torchmetrics

xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
# train model_B_on_A

In [20]:
#Momentum
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.05)

In [21]:
#Nesterov
optimizer = torch.optim.SGD(model.parameters(),
                            momentum=0.9, nesterov=True, lr=0.05)

In [22]:
#RMSProp
optimizer = torch.optim.RMSprop(model.parameters(), alpha=0.9, lr=0.05)

In [23]:
#Adam
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)

In [24]:
#AdamW
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

In [26]:
#Some model
# model = nn.Sequential(nn.Linear(10, 10))
#
# optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [27]:
# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         [...] the rest of the training loop remains unchanged
#
#     scheduler.step()

In [28]:
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20, eta_min=0.001)

In [29]:
# [...]  build the model and optimizer
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#   optimizer, mode="max", patience=2, factor=0.1)

In [30]:
# metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
# for epoch in range(n_epochs):
#     for X_batch, y_batch in train_loader:
#         [...]  the rest of the training loop remains unchanged
#     val_metric = evaluate_tm(model, valid_loader, metric).item()
#     scheduler.step(val_metric)

In [31]:
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)

In [32]:
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda epoch: (min(epoch, 3) / 3) * (1.0 - 0.1) + 0.1)

In [33]:
# for epoch in range(n_epochs):
#     warmup_scheduler.step()
#     for X_batch, y_batch in train_loader:
#         [...]   the rest of the training loop is unchanged
#     if epoch >= 3:   deactivate other scheduler(s) during warmup
#         scheduler.step(val_metric)

In [34]:
# cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer, T_0=2, T_mult=2, eta_min=0.001)